# Hamiltonian Flow Matching — Entropy Potential (2D)

Transports 8-Gaussians → Moons under the **entropy potential** `F(ρ) = β ∫ ρ log ρ dx`.

The variance σ(t) is governed by the ODE `σ' = -√(2β log(σ) + C)`, solved once at construction
via `DensityGaussianPath`. The mean follows a linear interpolation `μ_t = (1-t)x₀ + t x₁`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../../'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from torchdyn.core import NeuralODE

from torchcfm.hamiltonian import (
    EntropyPotential, DensityGaussianPath, HamiltonianFlowMatcher,
    flow_matching_loss,
)
from torchcfm.models.models_v2 import MLP
from torchcfm.utils import sample_8gaussians, sample_moons, torch_wrapper

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

dim        = 2
batch_size = 256
n_iters    = 20000
lr         = 1e-3

beta    = 2.0
coeff   = 0.75
sigma_0 = 0.15

print(f'device: {device}')
print(f'EntropyPotential: beta={beta}, coeff={coeff}, sigma_0={sigma_0}')

## Potential, path and flow matcher

`DensityGaussianPath` pre-computes the σ(t) schedule by integrating the entropy ODE
once at construction time (200 RK4 steps). No per-iteration solver calls.

In [ ]:
density_pot = EntropyPotential(beta=beta, coeff=coeff)
path        = DensityGaussianPath(density_pot, sigma_0=sigma_0, n_steps=200, method='rk4')
fm          = HamiltonianFlowMatcher(path, coupling='ot')

# Inspect the pre-computed schedule
sched = path.schedule
print(f'sigma schedule: t=[{sched.t[0]:.2f}, {sched.t[-1]:.2f}],  '
      f'sigma=[{sched.sigma[0].item():.4f}, {sched.sigma[-1].item():.4f}]')

In [ ]:
# Visualise σ(t)
t_np     = sched.t.numpy()
sigma_np = sched.sigma.squeeze(-1).numpy()

plt.figure(figsize=(6, 3))
plt.plot(t_np, sigma_np)
plt.xlabel('t'); plt.ylabel('σ(t)')
plt.title(f'Entropy potential: σ(t) schedule (beta={beta})')
plt.tight_layout(); plt.show()

In [ ]:
# σ'(t) — rate of variance change; negative means the distribution contracts toward the conditional mean
sigma_p_np = sched.sigma_prime.squeeze(-1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t_np, sigma_np)
axes[0].set_xlabel('t'); axes[0].set_ylabel('σ(t)')
axes[0].set_title(f'Entropy: σ(t)  (beta={beta})')

axes[1].plot(t_np, sigma_p_np, c='tab:orange')
axes[1].set_xlabel('t'); axes[1].set_ylabel("σ'(t)")
axes[1].set_title(f'Entropy: σ\'(t)  (beta={beta})')
plt.tight_layout(); plt.show()

## Model and training

In [ ]:
model     = MLP(dim + 1, out_dim=dim, w=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

model.train()
losses = []

for k in range(n_iters):
    optimizer.zero_grad()
    x0 = sample_8gaussians(batch_size).to(device)
    x1 = sample_moons(batch_size).to(device)

    t, xt, ut = fm.sample_location_and_conditional_flow(x0, x1)
    vt   = model(torch.cat([xt, t], dim=-1))
    loss = flow_matching_loss(vt, ut)

    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if k % 5000 == 0 or k == n_iters - 1:
        print(f'step {k:5d}: loss = {loss.item():.5f}')

In [ ]:
plt.figure(figsize=(7, 3))
plt.semilogy(losses)
plt.xlabel('iteration'); plt.ylabel('loss'); plt.title('Training loss')
plt.tight_layout(); plt.show()

## Evaluation

In [ ]:
model.eval()
node = NeuralODE(torch_wrapper(model), sensitivity='adjoint', solver='euler')

with torch.no_grad():
    traj = node.trajectory(
        sample_8gaussians(1000).to(device),
        t_span=torch.linspace(0, 1, 100, device=device),
    )

traj   = traj.cpu()
x1_ref = sample_moons(1000)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].scatter(traj[0, :, 0], traj[0, :, 1], s=3, alpha=0.5, c='steelblue')
axes[0].set_title('Source (8-Gaussians)')

for i in range(0, 1000, 5):
    axes[1].plot(traj[:, i, 0], traj[:, i, 1], alpha=0.05, c='gray', linewidth=0.5)
axes[1].set_title('Flow trajectories')

axes[2].scatter(traj[-1, :, 0], traj[-1, :, 1], s=3, alpha=0.5, c='tomato', label='generated')
axes[2].scatter(x1_ref[:, 0], x1_ref[:, 1], s=3, alpha=0.5, c='seagreen', label='target')
axes[2].legend(markerscale=3); axes[2].set_title('Generated vs Target')

plt.tight_layout(); plt.show()

In [ ]:
# Distribution snapshots at multiple time steps — shows how the 8-Gaussians mass disperses and
# rearranges toward the moons shape under the entropy-penalised flow
t_snaps = [0, 24, 49, 74, 99]
snap_labels = [f't = {i / 99:.2f}' for i in t_snaps]

fig, axes = plt.subplots(1, len(t_snaps), figsize=(16, 4), sharey=True)
for ax, ti, lbl in zip(axes, t_snaps, snap_labels):
    ax.scatter(traj[ti, :, 0].numpy(), traj[ti, :, 1].numpy(), s=4, alpha=0.6)
    ax.set_title(lbl); ax.set_xlim(-8, 8); ax.set_ylim(-6, 6); ax.set_aspect('equal')
plt.suptitle('Distribution evolution  (entropy potential)', fontsize=13)
plt.tight_layout(); plt.show()

# ρ₀ → ρ_t → ρ₁ scatter + trajectory lines
n_show = 1000
n_lines = 200

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(traj[0, :n_show, 0], traj[0, :n_show, 1],
                s=10, alpha=0.7, c='royalblue', label=r'$\rho_0$', zorder=3)
axes[0].scatter(traj[1:-1:5, :n_show, 0].reshape(-1), traj[1:-1:5, :n_show, 1].reshape(-1),
                s=0.3, alpha=0.15, c='black', label=r'$\rho_t$')
axes[0].scatter(traj[-1, :n_show, 0], traj[-1, :n_show, 1],
                s=10, alpha=0.7, c='tomato', label=r'$\rho_1$', zorder=3)
axes[0].legend(markerscale=4); axes[0].set_title(r'$\rho_0 \to \rho_t \to \rho_1$')
axes[0].set_xlim(-8, 8); axes[0].set_ylim(-6, 6)

for i in range(n_lines):
    axes[1].plot(traj[:, i, 0], traj[:, i, 1], 'gray', alpha=0.2, linewidth=0.6)
axes[1].scatter(traj[0, :n_lines, 0], traj[0, :n_lines, 1], s=6, c='royalblue', zorder=3)
axes[1].scatter(traj[-1, :n_lines, 0], traj[-1, :n_lines, 1], s=6, c='tomato', zorder=3)
axes[1].set_title('Trajectory paths'); axes[1].set_xlim(-8, 8); axes[1].set_ylim(-6, 6)
plt.suptitle('Entropy potential flow', fontsize=13)
plt.tight_layout(); plt.show()